In [1]:
# ==============================================================================
# CELL 1: SETUP AND SPARK SESSION
# ==============================================================================
!pip install pyspark==3.5.1 boto3

import os
import getpass
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, count, when, isnan, max, avg

# Configure AWS Credentials
print("Please enter your AWS Temporary Credentials.")
os.environ["AWS_ACCESS_KEY_ID"] = getpass.getpass("AWS Access Key ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass.getpass("AWS Secret Access Key: ")
os.environ["AWS_SESSION_TOKEN"] = getpass.getpass("AWS Session Token: ")

# Initialize Spark
spark = (
    SparkSession.builder
    .appName("Chunk_Validation_Checks")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.driver.memory", "7g")
    .config("spark.executor.memory", "3g")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.session.token", os.environ["AWS_SESSION_TOKEN"])
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .getOrCreate()
)
print(f"SparkSession created successfully. Spark Version: {spark.version}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 8.4 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=ba62d0395418a42f4834254d88c0436bd0711aba2a75df2bfa24a1f12917ac2a
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.3
    Uninstalling pyspark-4.0.3:
 

In [2]:
# ==============================================================================
# CELL 2: LOAD DATASET
# ==============================================================================
RUN_ID = "5f4e888a-66ee-4e42-a728-a4fa11bb8405"
S3A_PATH = f"s3a://ml-data-aws-transformer-bert/ml/chunking/chunked_documents/run_id={RUN_ID}/"

print(f"Loading chunked data from: {S3A_PATH}")
df_chunks = spark.read.parquet(S3A_PATH)

# Cache for faster downstream validation queries
df_chunks.cache()
total_chunks = df_chunks.count()
print(f"Total Chunks Loaded: {total_chunks:,}")

Loading chunked data from: s3a://ml-data-aws-transformer-bert/ml/chunking/chunked_documents/run_id=5f4e888a-66ee-4e42-a728-a4fa11bb8405/
Total Chunks Loaded: 688,122


In [3]:
# ==============================================================================
# CELL 3: CATEGORY DISTRIBUTION & INTEGRITY CHECKS
# ==============================================================================
print("--- Category Distribution ---")
df_chunks.groupBy("category").count().orderBy("category").show(truncate=False)

print("--- Null & Empty Value Checks ---")
# Dynamically check critical columns for nulls or empties
critical_cols = ["chunk_id", "parent_asin", "chunk_text", "product_title"]
null_exprs = [count(when(col(c).isNull() | isnan(c) | (col(c) == ""), c)).alias(c) for c in critical_cols]

df_chunks.select(*null_exprs).show(truncate=False)

--- Category Distribution ---
+-------------------+------+
|category           |count |
+-------------------+------+
|Appliances         |147467|
|Musical_Instruments|322372|
|Video_Games        |218283|
+-------------------+------+

--- Null & Empty Value Checks ---
+--------+-----------+----------+-------------+
|chunk_id|parent_asin|chunk_text|product_title|
+--------+-----------+----------+-------------+
|0       |0          |0         |45           |
+--------+-----------+----------+-------------+



In [4]:
# ==============================================================================
# CELL 4: CHUNK SEQUENCING & TEXT LENGTH METRICS
# ==============================================================================
print("--- Chunk Index Distribution per Category ---")
# Are there abnormally high chunk indexes?
df_chunks.groupBy("category").agg(
    max("chunk_index").alias("max_chunk_index"),
    avg("chunk_index").alias("avg_chunk_index")
).show(truncate=False)

print("--- Text Length Distribution (Character Count) ---")
# Validate that chunk_text is relatively uniform in size
df_length = df_chunks.withColumn("text_length", length(col("chunk_text")))

df_length.groupBy("category").agg(
    avg("text_length").alias("avg_chars_per_chunk"),
    max("text_length").alias("max_chars_per_chunk")
).show(truncate=False)

--- Chunk Index Distribution per Category ---
+-------------------+---------------+-------------------+
|category           |max_chunk_index|avg_chunk_index    |
+-------------------+---------------+-------------------+
|Video_Games        |3              |0.4220530229106252 |
|Musical_Instruments|4              |0.3957601776829253 |
|Appliances         |7              |0.45487464992167737|
+-------------------+---------------+-------------------+

--- Text Length Distribution (Character Count) ---
+-------------------+-------------------+-------------------+
|category           |avg_chars_per_chunk|max_chars_per_chunk|
+-------------------+-------------------+-------------------+
|Video_Games        |1493.176907958934  |2952               |
|Musical_Instruments|1366.328378395146  |2930               |
|Appliances         |1265.019807821411  |2991               |
+-------------------+-------------------+-------------------+



In [5]:
# ==============================================================================
# CELL 5: VISUAL SAMPLES BY CATEGORY
# ==============================================================================
categories = ["Appliances", "Musical_Instruments", "Video_Games"]

for cat in categories:
    print(f"\\n================ SAMPLE FOR: {cat.upper()} ================")
    (
        df_chunks.filter(col("category") == cat)
        .select("parent_asin", "chunk_index", "chunk_id", "chunk_text")
        .orderBy("parent_asin", "chunk_index")
        .show(3, truncate=90)
    )

\n================ SAMPLE FOR: APPLIANCES ================
+-----------+-----------+----------------------------------------------------------------+------------------------------------------------------------------------------------------+
|parent_asin|chunk_index|                                                        chunk_id|                                                                                chunk_text|
+-----------+-----------+----------------------------------------------------------------+------------------------------------------------------------------------------------------+
| 0967805929|          0|ffd210ed0b2defaa17737d246f74bd8ad0db005180c174fb2058f6f6abdff363|a more excellent way: be in health: pathways of wholeness, spiritual roots of disease t...|
| 0967805929|          1|375af01560ef35fe3b775a6b27b01e10373557102afaf72b92cda88a75317f93| detailed and fascinating, you may find it hard to put down. i do not agree with all it...|
| 1508810133|          0|ff52e0